In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm.notebook import tqdm
import math 
import numpy as np



In [18]:
movies_df = pd.read_csv('/content/movies.csv')
ratings_df = pd.read_csv('/content/ratings.csv')

# Merge the two DataFrames on 'movieId'
merged_df = pd.merge(ratings_df, movies_df, on='movieId', how='inner')

In [19]:
print("First 5 rows of the merged dataset:")
display(merged_df.head())

First 5 rows of the merged dataset:


,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [ ]:
# 1. Label Encoding for User IDs and Movie IDs
user_encoder = LabelEncoder()
merged_df['user_id_encoded'] = user_encoder.fit_transform(merged_df['userId'])

movie_encoder = LabelEncoder()
merged_df['movie_id_encoded'] = movie_encoder.fit_transform(merged_df['movieId'])

In [ ]:
# 2. Multi-Hot Encoding for Genres
# Fill any potential NaN values in 'genres' with an empty string before splitting
merged_df['genres'] = merged_df['genres'].fillna('')
# Split the genres string into a list of genres for each movie
merged_df['genre_list'] = merged_df['genres'].apply(lambda x: x.split('|') if x else [])

mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(merged_df['genre_list'])

genre_df = pd.DataFrame(genre_matrix, columns=mlb.classes_, index=merged_df.index)

In [ ]:
# 3. Merge everything into one clean DataFrame
processed_df = pd.concat([
    merged_df[['user_id_encoded', 'movie_id_encoded', 'rating']],
    genre_df
], axis=1)

print("First 5 rows of the processed DataFrame with encoded features:")
display(processed_df.head())

First 5 rows of the processed DataFrame with encoded features:


,user_id_encoded,movie_id_encoded,rating,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,0,4.0,0,0,1,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
1,0,2,4.0,0,0,0,0,0,1,0,...,0,0,0,0,0,1,0,0,0,0
2,0,5,4.0,0,1,0,0,0,0,1,...,0,0,0,0,0,0,0,1,0,0
3,0,43,5.0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0
4,0,46,5.0,0,0,0,0,0,0,1,...,0,0,0,0,1,0,0,1,0,0


In [23]:
# 4. Train/Test Split (80/20)
X = processed_df.drop('rating', axis=1) # Features (all columns except 'rating')
y = processed_df['rating'] # Target variable (the rating itself)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nShape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")


Shape of X_train: (80668, 22)
Shape of X_test: (20168, 22)
Shape of y_train: (80668,)
Shape of y_test: (20168,)


In [ ]:
# Custom Dataset class
class MovieLensDataset(Dataset):
    def __init__(self, user_ids, movie_ids, genres, ratings):
        self.user_ids = torch.LongTensor(user_ids.values)
        self.movie_ids = torch.LongTensor(movie_ids.values)
        # Ensure genres are float tensors for potential linear layers
        self.genres = torch.FloatTensor(genres.values)
        self.ratings = torch.FloatTensor(ratings.values)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.user_ids[idx], self.movie_ids[idx], self.genres[idx], self.ratings[idx]

In [ ]:
# Prepare data for the Dataset
# Separate features from X_train and X_test
X_train_users = X_train['user_id_encoded']
X_train_movies = X_train['movie_id_encoded']
X_train_genres = X_train.drop(columns=['user_id_encoded', 'movie_id_encoded'])

X_test_users = X_test['user_id_encoded']
X_test_movies = X_test['movie_id_encoded']
X_test_genres = X_test.drop(columns=['user_id_encoded', 'movie_id_encoded'])

# Create Dataset instances
train_dataset = MovieLensDataset(X_train_users, X_train_movies, X_train_genres, y_train)
test_dataset = MovieLensDataset(X_test_users, X_test_movies, X_test_genres, y_test)

# Initialize DataLoaders
batch_size = 256

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"\nNumber of training batches: {len(train_loader)}")
print(f"Number of testing batches: {len(test_loader)}")

print("\nSample batch from training DataLoader:")
for batch_idx, (users, movies, genres, ratings) in enumerate(train_loader):
    if batch_idx == 0:
        print(f"Users batch shape: {users.shape}")
        print(f"Movies batch shape: {movies.shape}")
        print(f"Genres batch shape: {genres.shape}")
        print(f"Ratings batch shape: {ratings.shape}")
        break


Number of training batches: 316
Number of testing batches: 79

Sample batch from training DataLoader:
Users batch shape: torch.Size([256])
Movies batch shape: torch.Size([256])
Genres batch shape: torch.Size([256, 20])
Ratings batch shape: torch.Size([256])


In [ ]:
# Get the number of unique users and movies for embedding dimensions
num_users = user_encoder.classes_.shape[0]
num_movies = movie_encoder.classes_.shape[0]
num_genres = X_train_genres.shape[1] 

class HybridNCF(nn.Module):
    def __init__(self, num_users, num_movies, num_genres, embedding_dim=32, hidden_layers=[128, 64, 32], dropout_rate=0.3):
        super(HybridNCF, self).__init__()

        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.movie_embedding = nn.Embedding(num_movies, embedding_dim)

        # 1. NEW: Bias Terms (This is the secret sauce for Popularity Bias)
        self.user_bias = nn.Embedding(num_users, 1)
        self.movie_bias = nn.Embedding(num_movies, 1)

        # 2. Genre Processing
        self.genre_dense = nn.Linear(num_genres, embedding_dim)

        # 3. Hidden Layers setup
        input_size = embedding_dim * 3 
        layers = []
        for i, h_size in enumerate(hidden_layers):
            layers.append(nn.Linear(input_size if i == 0 else hidden_layers[i-1], h_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate)) 

        self.hidden_layers = nn.Sequential(*layers)

        # 4. Output Layer
        self.output_layer = nn.Linear(hidden_layers[-1], 1)

    def forward(self, user_ids, movie_ids, genres):
        # Extract embeddings
        user_embed = self.user_embedding(user_ids)
        movie_embed = self.movie_embedding(movie_ids)

        # Extract biases and squeeze the extra dimension so they align with the output
        u_bias = self.user_bias(user_ids).squeeze(1)
        m_bias = self.movie_bias(movie_ids).squeeze(1)

        
        genre_features = F.relu(self.genre_dense(genres))

        # Concatenate and pass through dense layers
        combined_features = torch.cat((user_embed, movie_embed, genre_features), dim=1)
        x = self.hidden_layers(combined_features)
        
        base_prediction = self.output_layer(x).squeeze(1)

        # ADD THE BIASES to the final prediction
        final_prediction = base_prediction + u_bias + m_bias

        # Scale the output to a 0-5 range using Sigmoid
        prediction = torch.sigmoid(final_prediction) * 5

        return prediction

print("HybridNCF model class defined successfully!")

HybridNCF model class defined successfully!


In [ ]:
# Instantiate the model
model = HybridNCF(num_users, num_movies, num_genres, embedding_dim=50, hidden_layers=[128, 64, 32])

# Define Loss Function and Optimizer
criterion = nn.MSELoss() 
# Add weight_decay for L2 Regularization
# Updated optimizer with stronger weight decay
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
# Check for GPU availability and move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Using device: {device}")

num_epochs = 50

# Initialize for Early Stopping
best_loss = float('inf')
patience = 3 
patience_counter = 0 
best_model_path = 'best_recommender.pth' 

for epoch in range(num_epochs):
    model.train() 
    train_loss = 0.0

    # Use tqdm for a progress bar
    for user_ids, movie_ids, genres, ratings in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]"):
        # Move data to the same device as the model
        user_ids, movie_ids, genres, ratings = user_ids.to(device), movie_ids.to(device), genres.to(device), ratings.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(user_ids, movie_ids, genres)

        # Calculate loss
        loss = criterion(outputs, ratings)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * user_ids.size(0) 

    # Calculate average training loss for the epoch
    train_loss = train_loss / len(train_dataset)

    # Validation step
    model.eval() 
    test_loss = 0.0
    test_mae = 0.0 
    test_mse_sum = 0.0 # Initialize for RMSE calculation
    with torch.no_grad(): # Disable gradient calculation for validation
        for user_ids, movie_ids, genres, ratings in tqdm(test_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Test]"):
            user_ids, movie_ids, genres, ratings = user_ids.to(device), movie_ids.to(device), genres.to(device), ratings.to(device)
            outputs = model(user_ids, movie_ids, genres)

            # Calculate MSE loss
            loss = criterion(outputs, ratings)
            test_loss += loss.item() * user_ids.size(0)

            # Calculate MAE for evaluation metric
            batch_mae = F.l1_loss(outputs, ratings, reduction='sum').item()
            test_mae += batch_mae

            # Accumulate MSE for RMSE calculation
            test_mse_sum += loss.item() * user_ids.size(0)

    # Calculate average testing loss, MAE, and RMSE for the epoch
    test_loss = test_loss / len(test_dataset)
    test_mae = test_mae / len(test_dataset)
    test_rmse = math.sqrt(test_mse_sum / len(test_dataset))

    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}, Test MAE: {test_mae:.4f}, Test RMSE: {test_rmse:.4f}")

    # Early Stopping and Model Checkpointing
    if test_loss < best_loss:
        best_loss = test_loss
        patience_counter = 0 # Reset patience counter
        torch.save(model.state_dict(), best_model_path)
        print("New best model saved!")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs duek to no improvement in test loss for {patience} consecutive epochs.")
            break # Exit the training loop

print("Training complete!")

Using device: cpu


Epoch 1/50 [Train]:   0%|          | 0/316 [00:00<?, ?it/s]

Epoch 1/50 [Test]:   0%|          | 0/79 [00:00<?, ?it/s]

Epoch 1/50, Train Loss: 1.9635, Test Loss: 1.5292, Test MAE: 0.9565, Test RMSE: 1.2366
New best model saved!


Epoch 2/50 [Train]:   0%|          | 0/316 [00:00<?, ?it/s]

...

Epoch 17/50 [Test]:   0%|          | 0/79 [00:00<?, ?it/s]

Epoch 17/50, Train Loss: 0.5867, Test Loss: 0.7801, Test MAE: 0.6712, Test RMSE: 0.8832


Epoch 18/50 [Train]:   0%|          | 0/316 [00:00<?, ?it/s]

Epoch 18/50 [Test]:   0%|          | 0/79 [00:00<?, ?it/s]

Epoch 18/50, Train Loss: 0.5770, Test Loss: 0.7871, Test MAE: 0.6781, Test RMSE: 0.8872
Early stopping triggered after 18 epochs duek to no improvement in test loss for 3 consecutive epochs.
Training complete!


In [37]:
# Calculate overall popular movies for cold start scenario
# Group by movieId and calculate the mean rating and count of ratings
movie_stats = merged_df.groupby('movieId').agg(mean_rating=('rating', 'mean'), rating_count=('rating', 'count')).reset_index()

# Filter out movies with too few ratings to ensure robustness (e.g., minimum 50 ratings)
min_ratings_threshold = 50
qualified_movies = movie_stats[movie_stats['rating_count'] >= min_ratings_threshold]

# Sort by mean rating in descending order to get the most popular/highest rated
popular_movies_list_df = qualified_movies.sort_values(by='mean_rating', ascending=False)

# Merge with movies_df to get titles
popular_movies_list_df = pd.merge(popular_movies_list_df, movies_df, on='movieId', how='left')

# Get the top 5 movie titles and their average ratings
popular_movies_list = popular_movies_list_df[['title', 'mean_rating']].head(5)
print("Top 5 most popular movies (for cold start):")
display(popular_movies_list)

Top 5 most popular movies (for cold start):


,title,mean_rating
0,"Shawshank Redemption, The (1994)",4.429022
1,"Godfather, The (1972)",4.289062
2,Fight Club (1999),4.272936
3,Cool Hand Luke (1967),4.271930
4,Dr. Strangelove or: How I Learned to Stop Worr...,4.268041


In [ ]:
def get_top_recommendations(user_id_original, top_n=5):
    
    try:
        user_id_encoded = user_encoder.transform([user_id_original])[0]
    except ValueError:
        print(f"New user ID {user_id_original} detected! Returning top popular movies instead:")
        display(popular_movies_list)
        return

    # 1. Get all unique movie IDs that exist in our processed data
    all_movie_ids_encoded = processed_df['movie_id_encoded'].unique()

    # 2. Get movies already rated by the specified user
    
    rated_movies_by_user = merged_df[merged_df['userId'] == user_id_original]['movie_id_encoded'].unique()

    # 3. Identify movies the user has NOT rated
    unrated_movies_encoded = [mid for mid in all_movie_ids_encoded if mid not in rated_movies_by_user]

    if not unrated_movies_encoded:
        print(f"User {user_id_original} has rated all available movies or no unrated movies found.")
        return

    # 4. Prepare data for prediction
    # Create tensors for user_ids, movie_ids, and genres for unrated movies
    # The user_id will be constant for all predictions
    user_ids_tensor = torch.LongTensor([user_id_encoded] * len(unrated_movies_encoded)).to(device)
    movie_ids_tensor = torch.LongTensor(unrated_movies_encoded).to(device)

    # Get genre features for unrated movies
    # Need to map encoded movie IDs back to original movie IDs to get genre information
    # Create a temporary DataFrame to get genres for unrated movies efficiently
    temp_df = processed_df[processed_df['movie_id_encoded'].isin(unrated_movies_encoded)]
    # Ensure unique movie_id_encoded for genre_df to avoid duplicates in genre_features
    temp_df = temp_df.drop_duplicates(subset=['movie_id_encoded'])

    # Create a mapping from encoded movie ID to genre features
    # Ensure the order matches unrated_movies_encoded
    genre_features_list = []
    for encoded_movie_id in unrated_movies_encoded:
        # Get the row from temp_df corresponding to the current encoded_movie_id
        # Exclude 'user_id_encoded', 'movie_id_encoded', 'rating' to get only genre columns
        genre_row = temp_df[temp_df['movie_id_encoded'] == encoded_movie_id].drop(columns=['user_id_encoded', 'movie_id_encoded', 'rating'])
        genre_features_list.append(genre_row.values[0])

    # Convert list of numpy arrays to a single numpy array before creating tensor
    genres_tensor = torch.FloatTensor(np.array(genre_features_list)).to(device)

    # 6. Predict ratings using the trained model
    model.eval() # Set model to evaluation mode
    with torch.no_grad():
        predicted_ratings = model(user_ids_tensor, movie_ids_tensor, genres_tensor)

    # 7. Get the top N recommendations
    predicted_ratings = predicted_ratings.cpu().numpy() # Move to CPU and convert to numpy array

    # Create a DataFrame to sort and get top recommendations
    recommendations_df = pd.DataFrame({
        'movie_id_encoded': unrated_movies_encoded,
        'predicted_rating': predicted_ratings
    })

    recommendations_df = recommendations_df.sort_values(by='predicted_rating', ascending=False)

    # Map encoded movie_id back to original movie_id and then to title
    # Get original movie_id from the movie_encoder
    recommendations_df['movieId'] = recommendations_df['movie_id_encoded'].apply(lambda x: movie_encoder.inverse_transform([x])[0])

    # Merge with movies_df to get movie titles
    final_recommendations = pd.merge(recommendations_df, movies_df, on='movieId', how='left')

    print(f"\nTop {top_n} movie recommendations for User ID {user_id_original}:")
    display(final_recommendations[['title', 'predicted_rating']].head(top_n))

In [ ]:
def get_top_recommendations(user_id_original, top_n=5):
    # 1. Get all unique movie IDs that exist in our processed data
    all_movie_ids_encoded = processed_df['movie_id_encoded'].unique()

    # 2. Get movies already rated by the specified user
    # First, encode the original user_id to its encoded form
    try:
        user_id_encoded = user_encoder.transform([user_id_original])[0]
    except ValueError:
        print(f"User ID {user_id_original} not found in the dataset.")
        return

    # Filter the merged_df for the specific user
    rated_movies_by_user = merged_df[merged_df['userId'] == user_id_original]['movie_id_encoded'].unique()

    # 3. Identify movies the user has NOT rated
    unrated_movies_encoded = [mid for mid in all_movie_ids_encoded if mid not in rated_movies_by_user]

    if not unrated_movies_encoded:
        print(f"User {user_id_original} has rated all available movies or no unrated movies found.")
        return

    # 4. Prepare data for prediction
    # Create tensors for user_ids, movie_ids, and genres for unrated movies
    # The user_id will be constant for all predictions
    user_ids_tensor = torch.LongTensor([user_id_encoded] * len(unrated_movies_encoded)).to(device)
    movie_ids_tensor = torch.LongTensor(unrated_movies_encoded).to(device)

    # Get genre features for unrated movies
    temp_df = processed_df[processed_df['movie_id_encoded'].isin(unrated_movies_encoded)]
    
    temp_df = temp_df.drop_duplicates(subset=['movie_id_encoded'])

    # Create a mapping from encoded movie ID to genre features
    # Ensure the order matches unrated_movies_encoded
    genre_features_list = []
    for encoded_movie_id in unrated_movies_encoded:
        # Get the row from temp_df corresponding to the current encoded_movie_id
        # Exclude 'user_id_encoded', 'movie_id_encoded', 'rating' to get only genre columns
        genre_row = temp_df[temp_df['movie_id_encoded'] == encoded_movie_id].drop(columns=['user_id_encoded', 'movie_id_encoded', 'rating'])
        genre_features_list.append(genre_row.values[0])

    # Convert list of numpy arrays to a single numpy array before creating tensor
    genres_tensor = torch.FloatTensor(np.array(genre_features_list)).to(device)

    # 5. Predict ratings using the trained model
    model.eval() # Set model to evaluation mode
    with torch.no_grad():
        predicted_ratings = model(user_ids_tensor, movie_ids_tensor, genres_tensor)

    # 6. Get the top N recommendations
    predicted_ratings = predicted_ratings.cpu().numpy() # Move to CPU and convert to numpy array

    # Create a DataFrame to sort and get top recommendations
    recommendations_df = pd.DataFrame({
        'movie_id_encoded': unrated_movies_encoded,
        'predicted_rating': predicted_ratings
    })

    recommendations_df = recommendations_df.sort_values(by='predicted_rating', ascending=False)

    # Map encoded movie_id back to original movie_id and then to title
    # Get original movie_id from the movie_encoder
    recommendations_df['movieId'] = recommendations_df['movie_id_encoded'].apply(lambda x: movie_encoder.inverse_transform([x])[0])

    # Merge with movies_df to get movie titles
    final_recommendations = pd.merge(recommendations_df, movies_df, on='movieId', how='left')

    print(f"\nTop {top_n} movie recommendations for User ID {user_id_original}:")
    display(final_recommendations[['title', 'predicted_rating']].head(top_n))

# Example usage:
# Replace 'user_id_to_recommend_for' with an actual user ID from your dataset
user_id_to_recommend_for = 1 # Example user ID
get_top_recommendations(user_id_to_recommend_for)



Top 5 movie recommendations for User ID 1:


,title,predicted_rating
0,"Day of the Doctor, The (2013)",4.895804
1,Spotlight (2015),4.891139
2,Happiness (1998),4.880640
3,"Three Billboards Outside Ebbing, Missouri (2017)",4.873047
4,"Wind Rises, The (Kaze tachinu) (2013)",4.862624


In [ ]:
torch.save(model.state_dict(), 'recommender_model.pth')
print("Model weights saved to recommender_model.pth")

Model weights saved to recommender_model.pth


In [40]:
processed_df.to_csv('processed_data.csv', index=False)
print("processed_df saved to processed_data.csv")

processed_df saved to processed_data.csv
